# Customer Segmentation - Marketing Campaign
Cluster customers in `geniecode.default.marketing_campaign` on demographics and purchase behaviour (KMeans), then profile each segment including campaign response so marketing can target them.

* Clustering features: income, age, household, education, spend, channel mix, deal usage, product mix, recency, tenure
* Excluded from clustering (used only for profiling): `AcceptedCmp1-5`, `Response`, `Complain`; dropped: `ID`, constant `Z_CostContact`/`Z_Revenue`

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw = spark.table("geniecode.default.marketing_campaign").toPandas()
print("Shape:", raw.shape)
print("Missing values:\n", raw.isna().sum()[raw.isna().sum() > 0])
raw.info()
display(raw.head())
display(raw.describe().T.reset_index())

# Feature type overview
feature_types = pd.DataFrame({
    "feature": ["Year_Birth", "Education", "Marital_Status", "Income", "Kidhome / Teenhome", "Dt_Customer",
                "Recency", "Mnt* (6 product categories)", "Num*Purchases / NumWebVisitsMonth",
                "AcceptedCmp1-5 / Response", "Complain", "Z_CostContact / Z_Revenue", "ID"],
    "type": ["numeric integer", "categorical/ordinal", "categorical/nominal", "numeric float", "numeric integer (count)",
             "date", "numeric integer", "numeric integer (spend)", "numeric integer (count)",
             "binary (campaign outcome)", "binary", "constant (dropped)", "identifier (dropped)"]})
display(feature_types)

print(raw["Marital_Status"].value_counts())
print(raw["Education"].value_counts())

# Correlogram of numeric columns
num = raw.drop(columns=["ID", "Z_CostContact", "Z_Revenue"]).select_dtypes("number")
fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(num.corr(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num.columns))); ax.set_xticklabels(num.columns, rotation=90)
ax.set_yticks(range(len(num.columns))); ax.set_yticklabels(num.columns)
fig.colorbar(im); ax.set_title("Correlation matrix"); plt.tight_layout(); plt.show()

In [0]:
df = raw.copy()
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"])
ref_date = df["Dt_Customer"].max()

df["Age"] = ref_date.year - df["Year_Birth"]
df["Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Children"] = df["Kidhome"] + df["Teenhome"]
df["Has_Partner"] = df["Marital_Status"].isin(["Married", "Together"]).astype(int)
df["Edu_Level"] = df["Education"].map({"Basic": 0, "2n Cycle": 1, "Graduation": 2, "Master": 3, "PhD": 4})

mnt_cols = ["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df["Total_Spend"] = df[mnt_cols].sum(axis=1)
df["Total_Purchases"] = df[["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]].sum(axis=1)
df["Deal_Share"] = df["NumDealsPurchases"] / df["Total_Purchases"].replace(0, np.nan)
df["Wine_Share"] = df["MntWines"] / df["Total_Spend"]
df["Meat_Share"] = df["MntMeatProducts"] / df["Total_Spend"]
df["Gold_Share"] = df["MntGoldProds"] / df["Total_Spend"]
df["Campaigns_Accepted"] = df[[f"AcceptedCmp{i}" for i in range(1, 6)]].sum(axis=1)

# Remove implausible outliers: age > 90 (birth years 1893-1900) and income of 666,666
keep = (df["Age"] <= 90) & ((df["Income"] < 200_000) | df["Income"].isna())
print(f"Removed {(~keep).sum()} outlier rows")
df = df[keep].reset_index(drop=True)

# Keep only customers who purchased in the last 90 days (Recency = days since last purchase)
RECENCY_DAYS = 90
n_before = len(df)
df = df[df["Recency"] <= RECENCY_DAYS].reset_index(drop=True)
print(f"Recency filter (<= {RECENCY_DAYS} days): kept {len(df)} of {n_before} customers")

log_feats = ["Income", "Total_Spend", "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases",
             "NumDealsPurchases", "NumWebVisitsMonth"]          # heavy-tailed -> log1p
lin_feats = ["Age", "Children", "Has_Partner", "Edu_Level", "Recency", "Tenure_Days",
             "Deal_Share", "Wine_Share", "Meat_Share", "Gold_Share"]
features = log_feats + lin_feats
X = df[features]
print("Clustering on", len(features), "features,", len(X), "customers")

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

def make_preprocessor():
    return ColumnTransformer([
        ("log", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("sc", StandardScaler())]), log_feats),
        ("lin", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler())]), lin_feats)])

Xp = make_preprocessor().fit_transform(X)
ks, sils, inertias = range(2, 11), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xp)
    sils.append(silhouette_score(Xp, km.labels_)); inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inertias, marker="o"); axes[0].set_title("Elbow (inertia)"); axes[0].set_xlabel("k")
axes[1].plot(ks, sils, marker="o"); axes[1].set_title("Silhouette"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.show()
print(pd.DataFrame({"k": list(ks), "silhouette": np.round(sils, 3)}).to_string(index=False))

# k=2 has the top silhouette but only splits high vs low spenders. k=4 is the most
# actionable nearby option (balanced sizes, distinct profiles), so we use it.
K = 4

In [0]:
import mlflow
from mlflow.models import infer_signature

model = Pipeline([("pre", make_preprocessor()),
                  ("kmeans", KMeans(n_clusters=K, n_init=10, random_state=42))])
labels = model.fit_predict(X)
Xt = model.named_steps["pre"].transform(X)

sil = silhouette_score(Xt, labels)
dbi = davies_bouldin_score(Xt, labels)
ari = [adjusted_rand_score(labels, KMeans(n_clusters=K, n_init=10, random_state=s).fit_predict(Xt)) for s in (7, 123)]
sizes = np.bincount(labels)
print(f"Silhouette={sil:.3f}  Davies-Bouldin={dbi:.3f}  Seed-stability ARI={np.round(ari, 3)}")
print("Cluster sizes:", sizes, "| smallest share:", f"{sizes.min()/sizes.sum():.1%}")

with mlflow.start_run(run_name=f"customer_segmentation_kmeans_k{K}"):
    mlflow.log_params({"n_clusters": K, "n_init": 10, "random_state": 42, "features": ",".join(features),
                       "log_features": ",".join(log_feats), "n_rows": len(X)})
    mlflow.log_metrics({"silhouette": sil, "davies_bouldin": dbi, "ari_seed7": ari[0], "ari_seed123": ari[1]})
    sample = X.head(100)
    model_info = mlflow.sklearn.log_model(model, name="model",
                                          signature=infer_signature(sample, model.predict(sample)),
                                          input_example=X.head(3))
print("Logged model:", model_info.model_uri)

In [0]:
df["Cluster"] = labels
profile_cols = ["Income", "Age", "Children", "Has_Partner", "Edu_Level", "Total_Spend", "Total_Purchases",
                "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases", "NumDealsPurchases", "Deal_Share",
                "NumWebVisitsMonth", "Wine_Share", "Meat_Share", "Gold_Share", "Recency", "Tenure_Days",
                "Campaigns_Accepted", "Response", "Complain"]
prof = df.groupby("Cluster")[profile_cols].mean()

# Deterministic naming based on profile ranks (robust to label permutation)
by_spend = prof["Total_Spend"].sort_values(ascending=False).index.tolist()
low_two = prof.loc[by_spend[2:], "Age"].sort_values().index.tolist()
names = {by_spend[0]: "Affluent Premium Buyers",
         by_spend[1]: "Engaged Family Deal-Seekers",
         low_two[0]: "Young Budget Browsers",
         low_two[1]: "Budget-Conscious Families"}
df["Segment"] = df["Cluster"].map(names)

summary = df.groupby("Segment").agg(customers=("ID", "count"), **{c: (c, "mean") for c in profile_cols})
summary.insert(1, "share_pct", (100 * summary["customers"] / len(df)).round(1))
summary["spend_share_pct"] = (100 * df.groupby("Segment")["Total_Spend"].sum() / df["Total_Spend"].sum()).round(1)
summary = summary.sort_values("Total_Spend", ascending=False).round(2)
display(summary.reset_index())

# Campaign acceptance rate by segment (%)
cmp_cols = [f"AcceptedCmp{i}" for i in range(1, 6)] + ["Response"]
display((100 * df.groupby("Segment")[cmp_cols].mean()).round(1).reset_index())

# Top categorical makeup
display(pd.crosstab(df["Segment"], df["Education"], normalize="index").round(2).reset_index())

In [0]:
from sklearn.decomposition import PCA

pc = PCA(n_components=2, random_state=42).fit_transform(Xt)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for seg in summary.index:
    m = (df["Segment"] == seg).values
    axes[0].scatter(pc[m, 0], pc[m, 1], s=8, alpha=0.6, label=seg)
    axes[1].scatter(df.loc[m, "Income"], df.loc[m, "Total_Spend"], s=8, alpha=0.6, label=seg)
axes[0].set_title("PCA projection by segment"); axes[0].legend()
axes[1].set_title("Income vs Total Spend"); axes[1].set_xlabel("Income"); axes[1].set_ylabel("Total spend")

(100 * df.groupby("Segment")[["Response"]].mean()).loc[summary.index].plot.bar(ax=axes[2], legend=False)
axes[2].set_title("Last campaign response rate (%)"); axes[2].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

# Product mix by segment
mix = df.groupby("Segment")[mnt_cols].sum()
mix = mix.div(mix.sum(axis=1), axis=0).loc[summary.index]
mix.plot.barh(stacked=True, figsize=(10, 4), title="Product spend mix by segment"); plt.tight_layout(); plt.show()

## Segment findings - customers active in last 90 days (2,039 customers; KMeans k=4, silhouette 0.154, seed-stability ARI 0.994)

| Segment | Share | Profile | Last campaign response | Marketing play |
| --- | --- | --- | --- | --- |
| Affluent Premium Buyers | 27% (56% of spend) | Income ~75K, few kids, highest spend (~1,253), catalog + store, rarely use deals, few web visits | 26.0% (Cmp1 19.3%, Cmp5 23.0%) | Premium/loyalty offers, catalog, wine & meat bundles; no discounts needed |
| Engaged Family Deal-Seekers | 31% (38% of spend) | Income ~57K, ~1.2 kids, wine-heavy (62% of spend), heavy web + deal users, longest tenure | 14.7% (Cmp4 13.4%) | Web promotions, wine deals, family bundles, cross-sell meat/fish |
| Young Budget Browsers | 19% (3% of spend) | Youngest (~37), income ~26K, high gold share (26%), most web visits, deal-driven | 13.7% (Cmp3 10.9%) | Low-cost digital/mobile offers, gold/entry products, convert browsing to purchase |
| Budget-Conscious Families | 22% (3% of spend) | Most kids (~1.6), income ~40K, newest customers, low spend, deal-driven | 6.8% (lowest) | Low-priority for campaigns; nurture with value packs and onboarding |

In [0]:
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, HDBSCAN

# 1) Alternative algorithms vs the KMeans k=4 baseline
alt = []
for name, l in [("GMM k=4", GaussianMixture(4, covariance_type="full", random_state=42, n_init=3).fit(Xt).predict(Xt)),
                ("Ward k=4", AgglomerativeClustering(4, linkage="ward").fit_predict(Xt)),
                ("HDBSCAN mcs=25", HDBSCAN(min_cluster_size=25).fit_predict(Xt))]:
    alt.append({"method": name, "sizes": str(sorted(np.bincount(l + 1 if l.min() < 0 else l))),
                "ARI_vs_KMeans4": round(adjusted_rand_score(labels, l), 2)})
display(pd.DataFrame(alt))

# 2) KMeans with more clusters: stability and how current segments split
key_cols = ["Income", "Age", "Children", "Total_Spend", "NumCatalogPurchases", "NumWebPurchases",
            "Deal_Share", "Wine_Share", "Gold_Share", "Tenure_Days", "Response"]
for k in [5, 6, 7]:
    lk = KMeans(k, n_init=10, random_state=42).fit_predict(Xt)
    stab = [adjusted_rand_score(lk, KMeans(k, n_init=10, random_state=s).fit_predict(Xt)) for s in (7, 123)]
    print(f"k={k}: silhouette={silhouette_score(Xt, lk):.3f}, stability ARI={np.round(stab, 2)}")
    t = df.assign(cluster_k=lk)
    display(pd.crosstab(t["cluster_k"], t["Segment"]).reset_index())
    prof_k = t.groupby("cluster_k")[key_cols].mean().round(2)
    prof_k.insert(0, "n", t["cluster_k"].value_counts().sort_index())
    display(prof_k.reset_index())